# MM-Net — event-level respiratory metrics

The paper scores respiratory detection per 30-second epoch, and lists event-level scoring
as a limitation. A sleep technician does not think in epochs: they count *events*, and
they care how many false alarms an automated flag would generate per hour of recording.
This notebook computes both, so the limitation is quantified rather than merely stated.

**Definitions.** Within each patient, a true event is a maximal run of consecutive epochs
carrying a scored respiratory event. An event counts as *detected* if the model flags at
least one epoch inside it. A *false alarm* is a maximal run of flagged epochs that
overlaps no true event. False alarms are reported per hour of recording, at 120 epochs
per hour.

Events are computed **per patient**, never across concatenated patients, so no event is
created by a subject boundary.

In [ ]:
import json
import os
import sys
import time

import numpy as np

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
EPOCH_H = 120.0          # 30-second epochs per hour

def runs_of_ones(x):
    """Maximal runs of 1s in a binary vector, as (start, end_exclusive) pairs."""
    x = np.asarray(x).astype(bool)
    if not x.any():
        return []
    d = np.diff(np.concatenate(([0], x.view(np.int8), [0])))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))

print("device:", C.DEV)

## 1. Train the ten folds, keep per-patient scores

In [ ]:
t0 = time.time()
SUBJ = {}                     # sid -> (apnea_true, apnea_score)

for fi, (tr_all, te) in enumerate(C.FOLDS):
    rng = np.random.RandomState(100 + fi)
    tr_all = list(tr_all); rng.shuffle(tr_all)
    nv = max(10, len(tr_all) // 9)
    va, tr = tr_all[:nv], tr_all[nv:]
    model = C.train_fold(tr, va, "concat", [], [], seed=42)
    for s in te:
        _, apn = C.subj_infer(model, s, [], [])
        SUBJ[s] = (C.DATA[s][3].astype(int), apn)
    print("fold %d done (%.1f min)" % (fi, (time.time() - t0) / 60))

n_ev = sum(len(runs_of_ones(v[0])) for v in SUBJ.values())
n_ep = sum(len(v[0]) for v in SUBJ.values())
print("\npatients %d | epochs %d | scored events %d | %.1f h of recording"
      % (len(SUBJ), n_ep, n_ev, n_ep / EPOCH_H))

## 2. Event sensitivity and false alarms per hour

Swept across decision thresholds. Epoch-level sensitivity is shown alongside so the two
views can be compared directly: an event can be caught by flagging just one of its epochs,
so event sensitivity is expected to exceed epoch sensitivity.

In [ ]:
def evaluate(th):
    det = tot = fa = 0
    ep_tp = ep_pos = ep_flag = 0
    for yt, sc in SUBJ.values():
        pred = (sc >= th).astype(int)
        true_runs = runs_of_ones(yt)
        tot += len(true_runs)
        det += sum(1 for a, b in true_runs if pred[a:b].any())
        for a, b in runs_of_ones(pred):
            if not yt[a:b].any():
                fa += 1
        ep_tp += int(((pred == 1) & (yt == 1)).sum())
        ep_pos += int((yt == 1).sum())
        ep_flag += int((pred == 1).sum())
    hours = n_ep / EPOCH_H
    return dict(th=float(th),
                event_sens=det / tot if tot else float("nan"),
                fa_per_hour=fa / hours,
                epoch_sens=ep_tp / ep_pos if ep_pos else float("nan"),
                epoch_prec=ep_tp / ep_flag if ep_flag else float("nan"),
                flagged_frac=ep_flag / n_ep)

rows = [evaluate(t) for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7)]

print("%-8s %12s %14s %12s %12s %10s"
      % ("thresh", "event sens", "false alarms/h", "epoch sens", "epoch prec", "% flagged"))
print("-" * 74)
for r in rows:
    print("%-8.2f %12.3f %14.2f %12.3f %12.3f %10.1f"
          % (r["th"], r["event_sens"], r["fa_per_hour"], r["epoch_sens"],
             r["epoch_prec"], 100 * r["flagged_frac"]))

## 3. Per-patient agreement on event count

The clinically meaningful summary is not per-epoch accuracy but whether the model recovers
roughly the right *number* of events for a patient, since that is what an index is built
from.

In [ ]:
from scipy.stats import spearmanr

TH = 0.5
true_idx, pred_idx = [], []
for yt, sc in SUBJ.values():
    hours = len(yt) / EPOCH_H
    true_idx.append(len(runs_of_ones(yt)) / hours)
    pred_idx.append(len(runs_of_ones((sc >= TH).astype(int))) / hours)
true_idx, pred_idx = np.array(true_idx), np.array(pred_idx)

rho, p = spearmanr(true_idx, pred_idx)
print("per-patient event rate (events per hour), threshold %.2f" % TH)
print("  scored    mean %.1f  median %.1f  range %.1f-%.1f"
      % (true_idx.mean(), np.median(true_idx), true_idx.min(), true_idx.max()))
print("  predicted mean %.1f  median %.1f  range %.1f-%.1f"
      % (pred_idx.mean(), np.median(pred_idx), pred_idx.min(), pred_idx.max()))
print("  Spearman rho = %.3f (p = %.4f, n = %d)" % (rho, p, len(true_idx)))
print("  mean absolute error = %.1f events/hour" % np.abs(true_idx - pred_idx).mean())

json.dump({"threshold_sweep": rows,
           "event_rate": {"rho": float(rho), "p": float(p), "n": len(true_idx),
                          "mae": float(np.abs(true_idx - pred_idx).mean())}},
          open(os.path.join(OUT, "event_level_metrics.json"), "w"), indent=1)
print("\nwrote event_level_metrics.json")

## Reading the result

Event sensitivity will exceed epoch sensitivity, because catching one epoch of an event
counts as catching the event — that is the correct clinical view, and it is the more
flattering one. The number that keeps it honest is false alarms per hour: a flag a
technician would have to adjudicate several times an hour is not a labour saving. Reporting
both is the point, and it converts a stated limitation into a measured one.